In [18]:
import pandas as pd
import numpy as np

file_path = 'input_data/Source File 4-19.xlsx'

sheet_names = [
    "EDFT",
    "TH Summary",
    "TH Hourly",
    "ML Store",
    "Emp List",
    "Apr Unapr OT",
    "Check Pivot",
    "Current Week OT",
    "Sheet10",
    "SCH",
    "SCH for MO"
]

dfs = {}

for sheet in sheet_names:
    dfs[sheet] = pd.read_excel(file_path, sheet_name=sheet)



d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell X3100 is marked as a date but the serial value 3120000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell X3521 is marked as a date but the serial value 7150000 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell S18505 is marked as a date but the serial value 2102692565 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell S23019 is marked as a date but the serial value 7856725593 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Program Files\Python311\Lib\site-packages\ope

In [19]:
import time
# Access example:
df_edft = dfs["EDFT"]
df_th_summary = dfs["TH Summary"]
df_th_hourly = dfs["TH Hourly"]
df_ml_store = dfs["ML Store"]
# df_edft.head(10)


# ----------------------------
# PREP DATA
# ----------------------------
prep_start = time.time()
df = df_th_hourly.copy()

df_th_hourly["Date"] = pd.to_datetime(df_th_hourly["Date"], errors="coerce")
df_th_hourly["Hours"] = pd.to_numeric(df_th_hourly["Hours"], errors="coerce")



In [20]:
# Join TH Hourly with ML Store on Dist Department Code = Unique ID (case-insensitive)
# Create lowercase versions for matching
df_th_hourly["Dist Dept Code Lower"] = df_th_hourly["Dist Department Code"].astype(str).str.lower()
df_ml_store["Unique ID Lower"] = df_ml_store["Unique ID"].astype(str).str.lower()

df_th_hourly_joined = df_th_hourly.merge(
    df_ml_store,
    left_on="Dist Dept Code Lower",
    right_on="Unique ID Lower",
    how="left"
)

# Drop the lowercase columns after join
df_th_hourly_joined = df_th_hourly_joined.drop(columns=["Dist Dept Code Lower", "Unique ID Lower"])

# Filter only Pay Group = ML
df_th_hourly_joined_ml = df_th_hourly_joined[df_th_hourly_joined["Pay Group"] == "ML"]

df_th_hourly_joined_ml.to_csv("output_data/th_hourly_joined_ml.csv", index=False)

print(f"Original TH Hourly rows: {len(df_th_hourly)}")
print(f"After join & filter (ML only): {len(df_th_hourly_joined_ml)}")
print(f"Matched rows: {df_th_hourly_joined_ml['Unique ID'].notna().sum()}")
print("\nJoined columns:", df_th_hourly_joined_ml.columns.tolist())

Original TH Hourly rows: 17865
After join & filter (ML only): 14556
Matched rows: 14556

Joined columns: ['EE Code', 'Key', 'Employee_Name', 'Position_Title', 'UID', 'Pay Group', 'Employee_Status', 'DOL_Status', 'Pay_Type', 'Hire/Rehire Date', 'Home Department Code', 'Home Department Desc', 'Dist Department Code', 'Dist Department Desc', 'Day', 'Date', 'Earning Code', 'Earning Desc', 'Hours', 'Unique ID', 'Market Name', 'Store', 'SD Name', 'TM Name', 'HRBP / HRG', 'Tier', 'SOM Name', 'Store Contact', 'SOL Name']


In [21]:
# # Employee daily hours report (pivot: EE Code, Employee_Name as rows, Dates as columns)
# employee_daily_pivot = df_th_hourly_joined_ml.pivot_table(
#     index=["EE Code", "Employee_Name"],
#     columns="Date",
#     values="Hours",
#     aggfunc="sum",
#     fill_value=0
# )

# # Optional: Format date columns
# # employee_daily_pivot.columns = employee_daily_pivot.columns.strftime("%d-%b-%Y")

# # Add total hours column
# employee_daily_pivot["Total Hours"] = employee_daily_pivot.sum(axis=1)

# # Save to CSV
# employee_daily_pivot.to_csv("output_data/employee_daily_hours_pivot.csv")

# # Show sample
# employee_daily_pivot.head()

In [22]:
import copy
# Make a DEEP copy of df_th_hourly_joined_ml
df_th_hourly_copy = copy.deepcopy(df_th_hourly_joined_ml)

# Get actual min and max dates from data
min_date = df_th_hourly_copy["Date"].min()
max_date = df_th_hourly_copy["Date"].max()

# Add week column (Wednesday to Tuesday week)
df_th_hourly_copy["Week_Start"] = df_th_hourly_copy["Date"] - pd.to_timedelta(
    (df_th_hourly_copy["Date"].dt.weekday - 2) % 7, unit='D'
)
df_th_hourly_copy["Week_End"] = df_th_hourly_copy["Week_Start"] + pd.Timedelta(days=6)

# Adjust first week: if Week_Start is before min_date, use min_date
first_week_start = df_th_hourly_copy["Week_Start"].min()
if first_week_start < min_date:
    df_th_hourly_copy.loc[df_th_hourly_copy["Week_Start"] == first_week_start, "Week_Start"] = min_date
    df_th_hourly_copy.loc[df_th_hourly_copy["Week_Start"] == first_week_start, "Week_End"] = min_date + pd.Timedelta(days=6)

# Adjust last week: if Week_End is after max_date, use max_date
last_week_end = df_th_hourly_copy["Week_End"].max()
if last_week_end > max_date:
    df_th_hourly_copy.loc[df_th_hourly_copy["Week_End"] == last_week_end, "Week_End"] = max_date
    df_th_hourly_copy.loc[df_th_hourly_copy["Week_End"] == last_week_end, "Week_Start"] = max_date - pd.Timedelta(days=6)

df_th_hourly_copy["Week_Label"] = (
    df_th_hourly_copy["Week_Start"].dt.strftime("%d-%b-%Y") +
    " to " +
    df_th_hourly_copy["Week_End"].dt.strftime("%d-%b-%Y")
)

# Identify OT rows (based on Earning Desc containing 'OT')
df_th_hourly_copy["Is_OT"] = df_th_hourly_copy["Earning Desc"].astype(str).str.contains(
    "OT", case=False, na=False
)

# Get store columns (first row per EE Code - these are constant per employee)
store_cols = ["SD Name", "TM Name", "SOM Name", "Store Contact", "SOL Name", "Store"]
employee_info = df_th_hourly_copy.groupby("Key")[store_cols].first().reset_index()

# Calculate total hours and OT hours separately
total_hours = (
    df_th_hourly_copy
    .groupby(["Key", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Hours": "Total_Hours"})
)

ot_hours = (
    df_th_hourly_copy[df_th_hourly_copy["Is_OT"]]
    .groupby(["Key", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Hours": "OT_Hours"})
)

# Merge
weekly_report = total_hours.merge(
    ot_hours, on=["Key", "Week_Label"], how="left"
)
weekly_report["OT_Hours"] = weekly_report["OT_Hours"].fillna(0)

# Add store info
weekly_report = weekly_report.merge(employee_info, on="Key", how="left")

# Pivot: Week labels as columns
weekly_pivot = weekly_report.pivot_table(
    index=[
        "Key", "SD Name", "TM Name", "SOM Name",
        "Store Contact", "SOL Name", "Store"
    ],
    columns="Week_Label",
    values=["Total_Hours", "OT_Hours"],
    aggfunc="sum",
    fill_value=0
)

# Flatten column names
weekly_pivot.columns = [
    f"{col[1]}_{col[0]}" for col in weekly_pivot.columns
]

# Save to CSV
weekly_pivot.to_csv("output_data/employee_weekly_ot_report_pivot.csv")

# Show sample
weekly_pivot.head()

,,,,,,,01-Apr-2026 to 07-Apr-2026_OT_Hours,08-Apr-2026 to 14-Apr-2026_OT_Hours,15-Apr-2026 to 19-Apr-2026_OT_Hours,01-Apr-2026 to 07-Apr-2026_Total_Hours,08-Apr-2026 to 14-Apr-2026_Total_Hours,15-Apr-2026 to 19-Apr-2026_Total_Hours
Key,SD Name,TM Name,SOM Name,Store Contact,SOL Name,Store,,,,,,
A00870145313,Mohammed Dadani,Eloisa Perez,Heidi Gibson,Nimia Fernandez,Heidi Gibson SOL,Pleasant Valley,0.00,0.0,0.0,34.46,36.00,28.35
A00G70144329,David Brod,Hayden Robinson,Sonia Rajani,Lisa Saldana,Zubair Alam,Uvalde,0.00,0.0,0.0,36.21,34.32,19.63
A00K70144496,Zach Wagner,Courtney Guidry,Sandra Barragan,Perla Perez Laureano,Sandra Barragan SOL,Ambassador Rd,0.00,0.0,0.0,39.47,33.95,31.67
A00Z70144874,Marc Rivera,North Texas Market - No TM,Sonia Rajani,Arturo Lugo,Sonia Rajani SOL,Kiest Blvd,4.44,0.0,0.0,44.44,38.15,21.30
A01E70144944,Oh-PA Region - No SD,Carolina - West Market - No TM,Sonia Rajani,Helen Romero,Romana Soorty,Mount Airy,0.00,0.0,0.0,0.00,0.00,5.05


In [23]:
# OT hours per week per SOM Name
ot_by_som = (
    df_th_hourly_copy[df_th_hourly_copy["Is_OT"]]
    .groupby(["SOM Name", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Hours": "OT_Hours"})
)

# Pivot: SOM Name as rows, Week as columns
ot_by_som_pivot = ot_by_som.pivot_table(
    index="SOM Name",
    columns="Week_Label",
    values="OT_Hours",
    aggfunc="sum",
    fill_value=0
)

# Add total column
ot_by_som_pivot["Total_OT_Hours"] = ot_by_som_pivot.sum(axis=1)

# Save to CSV
ot_by_som_pivot.to_csv("output_data/ot_hours_by_som_week.csv")

# Show
ot_by_som_pivot

Week_Label,01-Apr-2026 to 07-Apr-2026,08-Apr-2026 to 14-Apr-2026,15-Apr-2026 to 19-Apr-2026,Total_OT_Hours
SOM Name,,,,
Heidi Gibson,40.75,50.14,0.00,90.89
Sandra Barragan,211.61,130.00,10.62,352.23
Sonia Rajani,287.07,252.62,29.30,568.99
Tashara League,233.71,130.82,1.40,365.93


In [24]:
# Biweekly OT Report by SOM Name
# Group weeks into biweekly periods (every 2 weeks)

# Get unique week labels in order
week_labels = ot_by_som["Week_Label"].unique()
print("Available weeks:", week_labels)

# Create biweek mapping with date ranges
# Group every 2 consecutive weeks into one biweek
# Extract start and end dates from week labels

def get_biweek_label(week_label, week_index, all_weeks):
    biweek_num = (week_index // 2) + 1
    
    # Get the two weeks in this biweek
    biweek_start_idx = (biweek_num - 1) * 2
    if biweek_start_idx >= len(all_weeks):
        return f"Biweek {biweek_num}"
    
    # Get first week of the biweek
    first_week = all_weeks[biweek_start_idx]
    
    # Try to extract dates from the week label
    try:
        # Format is "dd-Mon-yyyy to dd-Mon-yyyy"
        parts = first_week.split(" to ")
        if len(parts) == 2:
            start_date = parts[0]
            # Get the end date from the second week of the biweek
            if biweek_start_idx + 1 < len(all_weeks):
                second_week = all_weeks[biweek_start_idx + 1]
                second_parts = second_week.split(" to ")
                if len(second_parts) == 2:
                    end_date = second_parts[1]
                else:
                    end_date = parts[1]
            else:
                end_date = parts[1]
            return f"{start_date} to {end_date}"
    except:
        pass
    
    return f"Biweek {biweek_num}"

# Create a mapping from week to biweek with date ranges
week_to_biweek = {}
for i, week in enumerate(week_labels):
    week_to_biweek[week] = get_biweek_label(week, i, list(week_labels))

print("Week to Biweek mapping:", week_to_biweek)

# Add biweek column to ot_by_som
ot_by_som["Biweek"] = ot_by_som["Week_Label"].map(week_to_biweek)

# Aggregate OT hours by SOM Name and Biweek
ot_by_som_biweekly = (
    ot_by_som
    .groupby(["SOM Name", "Biweek"])["OT_Hours"]
    .sum()
    .reset_index()
)

# Pivot: SOM Name as rows, Biweek as columns
ot_by_som_biweekly_pivot = ot_by_som_biweekly.pivot_table(
    index="SOM Name",
    columns="Biweek",
    values="OT_Hours",
    aggfunc="sum",
    fill_value=0
)

# Add total column
# ot_by_som_biweekly_pivot["Total_OT_Hours"] = ot_by_som_biweekly_pivot.sum(axis=1)

# Save to CSV
# ot_by_som_biweekly_pivot.to_csv("output_data/ot_hours_by_som_biweek.csv")

# Show
ot_by_som_biweekly_pivot

Available weeks: <ArrowStringArray>
['01-Apr-2026 to 07-Apr-2026', '08-Apr-2026 to 14-Apr-2026',
 '15-Apr-2026 to 19-Apr-2026']
Length: 3, dtype: str
Week to Biweek mapping: {'01-Apr-2026 to 07-Apr-2026': '01-Apr-2026 to 14-Apr-2026', '08-Apr-2026 to 14-Apr-2026': '01-Apr-2026 to 14-Apr-2026', '15-Apr-2026 to 19-Apr-2026': '15-Apr-2026 to 19-Apr-2026'}


Biweek,01-Apr-2026 to 14-Apr-2026,15-Apr-2026 to 19-Apr-2026
SOM Name,,
Heidi Gibson,90.89,0.00
Sandra Barragan,341.61,10.62
Sonia Rajani,539.69,29.30
Tashara League,364.53,1.40


In [25]:
# Debug: Check breakdown by Earning Desc
print("Earning Desc value counts (ML only):")
print(df_th_hourly_joined_ml["Earning Desc"].value_counts())

print("\n--- OT Hours by Earning Desc ---")
ot_df = df_th_hourly_joined_ml[df_th_hourly_joined_ml["Earning Desc"].astype(str).str.contains("OT", case=False, na=False)]
print(ot_df.groupby("Earning Desc")["Hours"].sum())

Earning Desc value counts (ML only):
Earning Desc
Regular          14083
OT                 408
Paid Time Off       56
Bereavement          7
Sick                 1
Jury Duty            1
Name: count, dtype: int64

--- OT Hours by Earning Desc ---
Earning Desc
OT    1378.04
Name: Hours, dtype: float64


In [26]:
# Export to Excel with 2 tabs - Weekly and Biweekly
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.styles.numbers import FORMAT_NUMBER_00
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter

# Add total row to both dataframes
ot_by_som_pivot_with_total = ot_by_som_pivot.copy()
ot_by_som_pivot_with_total.loc["Total"] = ot_by_som_pivot_with_total.sum(numeric_only=True)

ot_by_som_biweekly_pivot_with_total = ot_by_som_biweekly_pivot.copy()
ot_by_som_biweekly_pivot_with_total.loc["Total"] = ot_by_som_biweekly_pivot_with_total.sum(numeric_only=True)

# Create a Excel writer
with pd.ExcelWriter("output_data/ot_pay_period_wise_ot_summary.xlsx", engine="openpyxl") as writer:
    # Write weekly data
    ot_by_som_pivot_with_total.to_excel(writer, sheet_name="Weekly")
    
    # Write biweekly data
    ot_by_som_biweekly_pivot_with_total.to_excel(writer, sheet_name="Biweekly")

print("Excel file saved: output_data/ot_pay_period_wise_ot_summary.xlsx")

# Now let's add formatting
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

wb = load_workbook("output_data/ot_pay_period_wise_ot_summary.xlsx")

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    
    # Header styling
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    header_alignment = Alignment(horizontal="center", vertical="center")
    
    # Total row styling
    total_font = Font(bold=True, color="000000")
    total_fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    
    thin_border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    # Format header row
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = thin_border
    
    # Format data cells and total row
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.border = thin_border
            if cell.column > 1:  # Number columns
                cell.alignment = Alignment(horizontal="right")
                cell.number_format = '0'  # Format to 0 decimal places
            # Check if this is the Total row
            if cell.row == ws.max_row:
                cell.font = total_font
                cell.fill = total_fill
    
    # Auto-adjust column widths
    for column in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        ws.column_dimensions[column_letter].width = adjusted_width

wb.save("output_data/ot_pay_period_wise_ot_summary.xlsx")
print("Formatted Excel file saved: output_data/ot_pay_period_wise_ot_summary.xlsx")

Excel file saved: output_data/ot_pay_period_wise_ot_summary.xlsx
Formatted Excel file saved: output_data/ot_pay_period_wise_ot_summary.xlsx


In [27]:
# Create df with latest week's OT
import copy

# df_th_hourly_joined_ml.to_csv("output_data/th_hourly_joined_ml.csv", index=False)
# Make a copy
df_ot = copy.deepcopy(df_th_hourly_joined_ml)

# Get min and max dates
min_date = df_ot["Date"].min()
max_date = df_ot["Date"].max()

# Add week columns (Wednesday to Tuesday week)
df_ot["Week_Start"] = df_ot["Date"] - pd.to_timedelta(
    (df_ot["Date"].dt.weekday - 2) % 7, unit='D'
)
df_ot["Week_End"] = df_ot["Week_Start"] + pd.Timedelta(days=6)

# Adjust first and last week as in the code
first_week_start = df_ot["Week_Start"].min()
if first_week_start < min_date:
    df_ot.loc[df_ot["Week_Start"] == first_week_start, "Week_Start"] = min_date
    df_ot.loc[df_ot["Week_Start"] == first_week_start, "Week_End"] = min_date + pd.Timedelta(days=6)

last_week_end = df_ot["Week_End"].max()
if last_week_end > max_date:
    df_ot.loc[df_ot["Week_End"] == last_week_end, "Week_End"] = max_date
    df_ot.loc[df_ot["Week_End"] == last_week_end, "Week_Start"] = max_date - pd.Timedelta(days=6)

df_ot["Week_Label"] = (
    df_ot["Week_Start"].dt.strftime("%d-%b-%Y") +
    " to " +
    df_ot["Week_End"].dt.strftime("%d-%b-%Y")
)

# Identify OT rows
df_ot["Is_OT"] = df_ot["Earning Desc"].astype(str).str.contains("OT", case=False, na=False)

# Find the latest week
latest_week = df_ot["Week_Label"].max()


# Filter for latest week and OT
df_latest_ot = df_ot[(df_ot["Is_OT"]) & (df_ot["Week_Label"] == latest_week)]

# df_latest_ot.to_csv("output_data/latest_week_ot_raw.csv", index=False)

# Select required columns
df_latest_ot_selected = df_latest_ot[[
    "Store",
    "SOL Name",
    "Employee_Name",
    "Position_Title",
    "UID",
    "Pay_Type",
    "Hours"
]].rename(columns={"Hours": "OT Hours", "SOL Name": "SOL"})

# Show each record
print("Latest week in data:", latest_week)
df_ot_current_week = df_latest_ot_selected
df_ot_current_week.to_csv("output_data/current_week_ot.csv", index=False)
# df_ot_current_week

Latest week in data: 15-Apr-2026 to 19-Apr-2026


In [28]:
# Employee-wise OT by Week with Home Location
import copy

# Make a copy
df_ot_e_wise = copy.deepcopy(df_th_hourly_joined_ml)

# Get min and max dates
min_date = df_ot_e_wise["Date"].min()
max_date = df_ot_e_wise["Date"].max()

# Add week columns (Wednesday to Tuesday week)
df_ot_e_wise["Week_Start"] = df_ot_e_wise["Date"] - pd.to_timedelta(
    (df_ot_e_wise["Date"].dt.weekday - 2) % 7, unit='D'
)
df_ot_e_wise["Week_End"] = df_ot_e_wise["Week_Start"] + pd.Timedelta(days=6)

# Adjust first and last week
first_week_start = df_ot_e_wise["Week_Start"].min()
if first_week_start < min_date:
    df_ot_e_wise.loc[df_ot_e_wise["Week_Start"] == first_week_start, "Week_Start"] = min_date
    df_ot_e_wise.loc[df_ot_e_wise["Week_Start"] == first_week_start, "Week_End"] = min_date + pd.Timedelta(days=6)

last_week_end = df_ot_e_wise["Week_End"].max()
if last_week_end > max_date:
    df_ot_e_wise.loc[df_ot_e_wise["Week_End"] == last_week_end, "Week_End"] = max_date
    df_ot_e_wise.loc[df_ot_e_wise["Week_End"] == last_week_end, "Week_Start"] = max_date - pd.Timedelta(days=6)

df_ot_e_wise["Week_Label"] = (
    df_ot_e_wise["Week_Start"].dt.strftime("%d") +
    " - " +
    df_ot_e_wise["Week_End"].dt.strftime("%d %B")
)

# Identify OT rows
df_ot_e_wise["Is_OT"] = df_ot_e_wise["Earning Desc"].astype(str).str.contains("OT", case=False, na=False)

# Filter for OT
df_ot_e_wise = df_ot_e_wise[df_ot_e_wise["Is_OT"]]

# Group by employee and week, sum OT hours
ot_e_wise_grouped = (
    df_ot_e_wise
    .groupby(["Home Department Desc", "SOM Name", "Employee_Name", "Home Department Code", "Week_Label"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Home Department Desc": "Store", "Hours": "OT_Hours", "SOM Name": "SOM", "Employee_Name": "Name"})
)

# Pivot: weeks as columns
ot_e_wise_ot_week_home_location = ot_e_wise_grouped.pivot_table(
    index=["Store", "SOM", "Name", "Home Department Code"],
    columns="Week_Label",
    values="OT_Hours",
    aggfunc="sum",
    fill_value=0
)

# Sort week columns in descending order (latest first)
week_cols = sorted(ot_e_wise_ot_week_home_location.columns, reverse=True)
ot_e_wise_ot_week_home_location = ot_e_wise_ot_week_home_location[week_cols]

# Reset index to make Store, SOM, Name, Home Department Code as columns
ot_e_wise_ot_week_home_location = ot_e_wise_ot_week_home_location.reset_index()

# Save to CSV
ot_e_wise_ot_week_home_location.to_csv("output_data/ot_e_wise_ot_week_home_location.csv", index=False)

# Show
ot_e_wise_ot_week_home_location

Week_Label,Store,SOM,Name,Home Department Code,15 - 19 April,08 - 14 April,01 - 07 April
0,2nd Ave,Sandra Barragan,"CHAREUNSOUK, DIAMOND MARYSE PIERRE",70144842,0.0,0.00,12.56
1,31st St,Sonia Rajani,"HERNANDEZ, JENNIFER JOCELYN",70144535,0.0,0.00,1.97
2,7th Street,Heidi Gibson,"GRIFFIN, CHOCOLET YVETTE",70145666,0.0,0.00,2.09
3,8711 N Lamar,Heidi Gibson,"HERNANDEZ, ROCIO",70145356,0.0,0.00,0.65
4,Abilene,Sonia Rajani,"UTT, ROCIO",70144436,0.0,12.47,8.12
...,...,...,...,...,...,...,...
227,Wilmington,Tashara League,"GARDUNO, MIA RAFAELA",70144718,0.0,4.87,4.01
228,Wilson,Tashara League,"ASCENCIO, BRIAN",70144467,0.0,0.00,0.52
229,Winchester Rd,Sandra Barragan,"MONZON, RANDI LEIGH",70164422,0.0,0.00,8.13
230,Wisconsin Rapids,Tashara League,"PARKINSON, ASHLEY LEIGH",70144572,0.0,0.48,2.80


In [29]:
# Employee-wise OT by Pay Period (Biweekly) with Home Location
import copy

# Make a copy
df_ot_e_wise_pp = copy.deepcopy(df_th_hourly_joined_ml)

# Get min and max dates
min_date = df_ot_e_wise_pp["Date"].min()
max_date = df_ot_e_wise_pp["Date"].max()

# Add week columns (Wednesday to Tuesday week)
df_ot_e_wise_pp["Week_Start"] = df_ot_e_wise_pp["Date"] - pd.to_timedelta(
    (df_ot_e_wise_pp["Date"].dt.weekday - 2) % 7, unit='D'
)
df_ot_e_wise_pp["Week_End"] = df_ot_e_wise_pp["Week_Start"] + pd.Timedelta(days=6)

# Adjust first and last week
first_week_start = df_ot_e_wise_pp["Week_Start"].min()
if first_week_start < min_date:
    df_ot_e_wise_pp.loc[df_ot_e_wise_pp["Week_Start"] == first_week_start, "Week_Start"] = min_date
    df_ot_e_wise_pp.loc[df_ot_e_wise_pp["Week_Start"] == first_week_start, "Week_End"] = min_date + pd.Timedelta(days=6)

last_week_end = df_ot_e_wise_pp["Week_End"].max()
if last_week_end > max_date:
    df_ot_e_wise_pp.loc[df_ot_e_wise_pp["Week_End"] == last_week_end, "Week_End"] = max_date
    df_ot_e_wise_pp.loc[df_ot_e_wise_pp["Week_End"] == last_week_end, "Week_Start"] = max_date - pd.Timedelta(days=6)

df_ot_e_wise_pp["Week_Label"] = (
    df_ot_e_wise_pp["Week_Start"].dt.strftime("%d-%b-%Y") +
    " to " +
    df_ot_e_wise_pp["Week_End"].dt.strftime("%d-%b-%Y")
)

# Identify OT rows
df_ot_e_wise_pp["Is_OT"] = df_ot_e_wise_pp["Earning Desc"].astype(str).str.contains("OT", case=False, na=False)

# Filter for OT
df_ot_e_wise_pp = df_ot_e_wise_pp[df_ot_e_wise_pp["Is_OT"]]

# Get unique week labels in order
week_labels = df_ot_e_wise_pp["Week_Label"].unique()
week_labels_sorted = sorted(week_labels)

# Create pay period mapping (every 2 weeks)
def get_pay_period_label(week_label, week_index, all_weeks):
    pp_num = (week_index // 2) + 1
    
    # Get the two weeks in this pay period
    pp_start_idx = (pp_num - 1) * 2
    if pp_start_idx >= len(all_weeks):
        return f"Pay Period {pp_num}"
    
    # Get first week of the pay period
    first_week = all_weeks[pp_start_idx]
    
    # Try to extract dates from the week label
    try:
        # Format is "dd-Mon-yyyy to dd-Mon-yyyy"
        parts = first_week.split(" to ")
        if len(parts) == 2:
            start_date_str = parts[0]
            start_date = pd.to_datetime(start_date_str)
            # Get the end date from the second week of the pay period
            if pp_start_idx + 1 < len(all_weeks):
                second_week = all_weeks[pp_start_idx + 1]
                second_parts = second_week.split(" to ")
                if len(second_parts) == 2:
                    end_date_str = second_parts[1]
                    end_date = pd.to_datetime(end_date_str)
                else:
                    end_date_str = parts[1]
                    end_date = pd.to_datetime(end_date_str)
            else:
                end_date_str = parts[1]
                end_date = pd.to_datetime(end_date_str)
            return f"Pay Period {start_date.strftime('%d')} - {end_date.strftime('%d %b %Y')}"
    except:
        pass
    
    return f"Pay Period {pp_num}"

# Create a mapping from week to pay period
week_to_pp = {}
for i, week in enumerate(week_labels_sorted):
    week_to_pp[week] = get_pay_period_label(week, i, week_labels_sorted)

# Add pay period column
df_ot_e_wise_pp["Pay_Period"] = df_ot_e_wise_pp["Week_Label"].map(week_to_pp)

# Group by employee and pay period, sum OT hours
ot_e_wise_pp_grouped = (
    df_ot_e_wise_pp
    .groupby(["Home Department Desc", "TM Name", "Employee_Name", "Position_Title", "Home Department Code", "Pay_Period"])["Hours"]
    .sum()
    .reset_index()
    .rename(columns={"Home Department Desc": "Store", "Hours": "OT_Hours", "TM Name": "TM", "Employee_Name": "Name", "Position_Title": "Role"})
)

# Pivot: pay periods as columns
ot_e_wise_ot_pp_home_location = ot_e_wise_pp_grouped.pivot_table(
    index=["Store", "TM", "Name", "Role", "Home Department Code"],
    columns="Pay_Period",
    values="OT_Hours",
    aggfunc="sum",
    fill_value=0
)

# Sort pay period columns in descending order (latest first)
pp_cols = sorted(ot_e_wise_ot_pp_home_location.columns, reverse=True)
ot_e_wise_ot_pp_home_location = ot_e_wise_ot_pp_home_location[pp_cols]

# Reset index to make Store, TM, Name, Role, Home Department Code as columns
ot_e_wise_ot_pp_home_location = ot_e_wise_ot_pp_home_location.reset_index()

# Save to CSV
ot_e_wise_ot_pp_home_location.to_csv("output_data/ot_e_wise_ot_pp_home_location.csv", index=False)

# Show
ot_e_wise_ot_pp_home_location

Pay_Period,Store,TM,Name,Role,Home Department Code,Pay Period 15 - 19 Apr 2026,Pay Period 01 - 14 Apr 2026
0,2nd Ave,Alabama North Market - No TM,"CHAREUNSOUK, DIAMOND MARYSE PIERRE",Bilingual Sales Advocate,70144842,0.0,12.56
1,31st St,Justin Kaiser,"HERNANDEZ, JENNIFER JOCELYN",Bilingual Acting Store Manager,70144535,0.0,1.97
2,7th Street,Alex Bunton,"GRIFFIN, CHOCOLET YVETTE",Bilingual Sales Advocate,70145666,0.0,2.09
3,8711 N Lamar,Erin Dela Garza,"HERNANDEZ, ROCIO",Bilingual Acting Store Manager,70145356,0.0,0.65
4,Abilene,Venessa Parker,"UTT, ROCIO",Bilingual Retail Store Manager I,70144436,0.0,20.59
...,...,...,...,...,...,...,...
227,Wilmington,Mariela Acosta-Santiago,"GARDUNO, MIA RAFAELA",Bilingual Sales Advocate,70144718,0.0,8.88
228,Wilson,Christian Toledo,"ASCENCIO, BRIAN",Bilingual Store Lead,70144467,0.0,0.52
229,Winchester Rd,Montavious Mckinney,"MONZON, RANDI LEIGH",Bilingual Sales Advocate,70164422,0.0,8.13
230,Wisconsin Rapids,Lucero Sanchez,"PARKINSON, ASHLEY LEIGH",Sales Advocate,70144572,0.0,3.28


In [40]:
df_sch = dfs["SCH"]

df_sch["Type"] = df_sch["Type"].fillna("").str.strip()

df_sch_pivot = df_sch.pivot_table(
    index=["EE Code", "UID","Date"],
    columns="Type",
    values="TH",
    aggfunc="sum",   # important if duplicates exist
    fill_value=0
).reset_index()

df_sch_pivot.to_csv("output_data/sch_pivot.csv", index=False)

print(week_to_pp)

{'01-Apr-2026 to 07-Apr-2026': 'Pay Period 01 - 14 Apr 2026', '08-Apr-2026 to 14-Apr-2026': 'Pay Period 01 - 14 Apr 2026', '15-Apr-2026 to 19-Apr-2026': 'Pay Period 15 - 19 Apr 2026'}


In [41]:
df_sch_pivot["Date"] = pd.to_datetime(df_sch_pivot["Date"], dayfirst=True)

records = []

for week_range, pp in week_to_pp.items():
    start, end = week_range.split(" to ")
    
    records.append({
        "Pay_Period": pp,
        "Start_Date": pd.to_datetime(start, dayfirst=True),
        "End_Date": pd.to_datetime(end, dayfirst=True)
    })

df_pp = pd.DataFrame(records)

def get_pay_period(date):
    for _, row in df_pp.iterrows():
        if row["Start_Date"] <= date <= row["End_Date"]:
            return row["Pay_Period"]
    return None

df_sch_with_pp = df_sch_pivot.copy()


df_sch_with_pp["Pay_Period"] = df_sch_with_pp["Date"].apply(get_pay_period)

df_sch_with_pp = df_sch_with_pp.groupby(
    ["EE Code", "UID", "Pay_Period"]
)[["Break", "Lunch", "Sch"]].sum().reset_index()

df_sch_with_pp.to_csv("output_data/sch_with_pay_period.csv", index=False)